In [1]:
import boto3
import json
bedrock_client = boto3.client(service_name = "bedrock" , region_name = "us-east-1")
bedrock_runtime_client = boto3.client(service_name = "bedrock-runtime" ,region_name = "us-east-1" )
bedrock_agent_client = boto3.client(service_name ="bedrock-agent", region_name = "us-east-1")
bedrock_agent_runtime_client = boto3.client(service_name= "bedrock-agent-runtime" ,region_name = "us-east-1")
models = [{"modelArn" : "arn:aws:bedrock:us-east-1:546203070602:inference-profile/us.anthropic.claude-3-haiku-20240307-v1:0"},
          {"modelArn": "arn:aws:bedrock:us-east-1:546203070602:inference-profile/us.anthropic.claude-3-5-sonnet-20240620-v1:0"}]
model_id = 'amazon.nova-micro-v1:0'
lambda_client = boto3.client(service_name = "lambda" , region_name = "us-east-1")

In [23]:
routing_criteria = {"responseQualityDifference":10}
response = bedrock_client.create_prompt_router(
    models = models,
    promptRouterName = "Custom-router-1",
    routingCriteria = routing_criteria,
    fallbackModel = models[1]
)
response

ValidationException: An error occurred (ValidationException) when calling the CreatePromptRouter operation: Failed to create the resource because the prompt router with name Custom-router-1 already exists.

In [24]:
cachePoint = {"cachePoint":{"type": "default"}}
cache_query = {
    "system":[
        {"text" :"You are a helpful assistant which summarizes what the user says?"},
        cachePoint,
    ],
    "messages":[
        {"role":"user","content":[{"text":"This thing isn't helpful at all.I have never expected this from the company.I'am very disappointed"}]}
    ]
}
response = bedrock_runtime_client.converse(
    modelId = 'amazon.nova-micro-v1:0',
    system = cache_query["system"],
    messages = cache_query["messages"]
)
print(response["output"]["message"]["content"][0]["text"])

It sounds like you are very disappointed with a product or service from the company and feel it hasn't met your expectations at all. You seem to be expressing frustration and let down by the experience.


In [25]:
try:
    system_prompt = bedrock_agent_client.create_prompt(
        name = "Review-summarizer",
        description = "Gives the summary of the review submitted by the user",
        variants = [
            {
                "name": "VariantV1",
                "modelId": model_id,
                "templateType": "CHAT",
                "inferenceConfiguration":{
                    "text":{
                        "temperature":0.7
                    }
                },
                "templateConfiguration":{
                    "chat":{
                        "system":[
                            {"text": """You are a helpful assistant.Your job is to ensure you summarize the review properly.
                            Apart from summarizing ,you have to also tell
                            A. Are there any swear or insulting words.
                            B. Is there any personally identifiable information if the user"""},
                            cachePoint
                        ],
                        "messages":[{
                            "role": "user","content":[{"text":"Review {{info}}"}]
                        }],
                        "inputVariables":[
                            {"name":"info"}
                        ]
                    }
                }
            }
        ]
    )
    print("Prompt created")
    prompt_arn = system_prompt["arn"]
except bedrock_runtime_client.exceptions.ConflictException as e:
    print("Already exists!")
    response = bedrock_runtime_client.list_prompts()
    prompt = next((prompt for prompt in system_prompt ['promptSummaries'] if prompt['name'] == "Review-summarizer"), None)
    prompt_arn = prompt['arn']
prompt_arn    

ConflictException: An error occurred (ConflictException) when calling the CreatePrompt operation: Couldn't perform CreatePrompt operation. The name Review-summarizer already exists for id XHW8WCDBS9. Retry your request with a different name.

In [26]:
response = bedrock_runtime_client.converse_stream(
    modelId = prompt_arn,
    promptVariables = {
        "info":{
            "text":""" What a stupid thing i ever saw.Totally not worth my money.Please rfund my money
            """
        }
    }
)
for event in response["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"]["text"])

**
Summary:**
The review expresses
 strong
 dissatisfaction with a
 product, describing it as "
stupid"
 and stating that it was
 not worth the
 money spent. The reviewer is
 requesting a refund.


**Analysis
:**
A
. Yes
, there are swear or
 insulting words.
 The term
 "stupid" is used to
 describe the product in a derogatory
 manner.
B. No,
 there is
 no personally identifiable information about
 the user in
 the review.


In [26]:
response = bedrock_runtime_client.converse_stream(
    modelId = prompt_arn,
    promptVariables = {
        "info":{
            "text":""" What a stupid thing i ever saw.Totally not worth my money.Please rfund my money
            """
        }
    }
)
for event in response["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"]["text"])

**
Summary:**
The review expresses
 strong
 dissatisfaction with a
 product, describing it as "
stupid"
 and stating that it was
 not worth the
 money spent. The reviewer is
 requesting a refund.


**Analysis
:**
A
. Yes
, there are swear or
 insulting words.
 The term
 "stupid" is used to
 describe the product in a derogatory
 manner.
B. No,
 there is
 no personally identifiable information about
 the user in
 the review.


In [26]:
response = bedrock_runtime_client.converse_stream(
    modelId = prompt_arn,
    promptVariables = {
        "info":{
            "text":""" What a stupid thing i ever saw.Totally not worth my money.Please rfund my money
            """
        }
    }
)
for event in response["stream"]:
    if "contentBlockDelta" in event:
        print(event["contentBlockDelta"]["delta"]["text"])

**
Summary:**
The review expresses
 strong
 dissatisfaction with a
 product, describing it as "
stupid"
 and stating that it was
 not worth the
 money spent. The reviewer is
 requesting a refund.


**Analysis
:**
A
. Yes
, there are swear or
 insulting words.
 The term
 "stupid" is used to
 describe the product in a derogatory
 manner.
B. No,
 there is
 no personally identifiable information about
 the user in
 the review.


In [43]:
LAMBDA_TOOLS = {
    "add_numbers":"add_numbers",
    "multiply_numbers": "multiply_numbers"
}
    

In [57]:
math_tool = [
    {"toolSpec":{
        "name":"add_numbers",
        "description":" Adds 2 numbers",
        "inputSchema":{
            "json":{
                "type":"object",
                "properties":{
                "num1":{"type": "number"},
                "num2":{"type": "number"}
                },
                "required":["num1" , "num2"]
                
            }
        }
     }
    },
    {
    "toolSpec":{
        "name":"multiply_numbers",
        "description": "Multiplies 2 numbers and returns the result",
        "inputSchema":{
            "json":{
                "type": "object",
                "properties":{
                    "num1":{"type":"number"},
                    "num2": {"type": "number"}
                },
                "required" : ["num1" , "num2"]
            }
        }
    }
    }      
]

In [58]:
def execute_lamda_tools(tool_name , tool_input):
    lambda_function =  LAMBDA_TOOLS[tool_name]
    response = lambda_client.invoke(
        FunctionName = lambda_function,
        InvocationType = "RequestResponse",
        Payload = json.dumps(tool_input)
    )
    return json.loads(response["Payload"].read())

In [66]:
messages = [{"role":"user", "content":[{"text":"What happens when we multiply 5 and 6."}]}]
tool_interaction = bedrock_runtime_client.converse(
    modelId = model_id,
    messages = messages,
    toolConfig ={
       "tools":math_tool,
        "toolChoice":{"auto":{}}
    },
    inferenceConfig ={
        "temperature" :0.7
    }
)
assistant_reply = tool_interaction["output"]["message"]
messages_parts = assistant_reply["content"]
tool_block_request = next((part for part in messages_parts if "toolUse" in part),None)
tool_block_request
print("/n Executing the tool")
tool_name = tool_block_request["toolUse"]["name"]
tool_input = tool_block_request["toolUse"]["input"]
tool_use_id = tool_block_request["toolUse"]["toolUseId"]

/n Executing the tool


In [69]:
#Calling the tool
result = execute_lamda_tools(tool_name , tool_input)
messages.append({
    "role": "assistant",
    "content":messages_parts
})
messages.append({
    "role": "user",
    "content":[
        {
            "toolResult":{
                "toolUseId": tool_use_id,
                "content":[{"text":str(result)}]
            }
        }
        
    ]
})
messages

[{'role': 'user',
  'content': [{'text': 'What happens when we multiply 5 and 6.'}]},
 {'role': 'assistant',
  'content': [{'text': '<thinking> The User wants to know the result of multiplying 5 and 6. I can use the "multiply_numbers" tool to get this result. </thinking>\n'},
   {'toolUse': {'toolUseId': 'tooluse_MNGA4URUEaZ8no8CIhCtaz',
     'name': 'multiply_numbers',
     'input': {'num1': 5, 'num2': 6}}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_MNGA4URUEaZ8no8CIhCtaz',
     'content': [{'text': '30'}]}}]},
 {'role': 'assistant',
  'content': [{'text': '<thinking> The User wants to know the result of multiplying 5 and 6. I can use the "multiply_numbers" tool to get this result. </thinking>\n'},
   {'toolUse': {'toolUseId': 'tooluse_MNGA4URUEaZ8no8CIhCtaz',
     'name': 'multiply_numbers',
     'input': {'num1': 5, 'num2': 6}}}]},
 {'role': 'user',
  'content': [{'toolResult': {'toolUseId': 'tooluse_MNGA4URUEaZ8no8CIhCtaz',
     'content': [{'text': '3

In [71]:
print("\n📤 Getting final response from Bedrock...")
final_response = bedrock_runtime_client.converse(
    modelId=model_id,
    messages=messages,
    toolConfig={"tools": math_tool, "toolChoice": {"auto": {}}},
    inferenceConfig={"temperature": 0.7}
)

final_answer = final_response["output"]["message"]["content"][0]["text"]
final_answer


📤 Getting final response from Bedrock...


'\nThe result of multiplying 5 and 6 is 30.'

In [72]:
input_text = "I'am creating some dummy text for embeddings."
response = bedrock_runtime_client.invoke_model(
    modelId = "amazon.titan-embed-text-v2:0",
    body = json.dumps({
        "inputText": input_text,
        "dimensions": 512,
        "normalize": True
    })
)
response_body = json.loads(response['body'].read())
embedding = response_body['embedding']
embedding

[-0.11361600458621979,
 0.1042315736413002,
 0.017806706950068474,
 -0.0027425845619291067,
 0.04697355255484581,
 0.0654037594795227,
 0.061479050666093826,
 -0.030111612752079964,
 0.012024084106087685,
 -0.01066746935248375,
 -0.019141776487231255,
 -0.005485924892127514,
 0.025833135470747948,
 -0.03173733130097389,
 -0.06033509597182274,
 0.05707458779215813,
 -0.03214855119585991,
 0.0300874225795269,
 0.058859553188085556,
 0.016892679035663605,
 0.023427307605743408,
 0.03569933399558067,
 0.04745481535792351,
 -0.033718839287757874,
 -0.005404286552220583,
 0.04129209741950035,
 0.028716696426272392,
 -0.04697682335972786,
 0.021427664905786514,
 -0.030604468658566475,
 0.03885502740740776,
 0.0016816583229228854,
 0.0016690597403794527,
 -0.026434842497110367,
 0.06185196712613106,
 0.09111544489860535,
 0.010799502953886986,
 0.010003775358200073,
 -0.019720304757356644,
 -0.01781942881643772,
 0.012805195525288582,
 0.029773781076073647,
 -0.0007130795856937766,
 0.08483605

In [4]:
agent = bedrock_agent_client.create_agent(
    agentName = "DummyAgent",
    description = "A dummy agent to test whether agent creation works or not",
    instruction = "You are a dummy agent.Your job is to do nothing",
    foundationModel = model_id
)
print(agent)

{'ResponseMetadata': {'RequestId': '991a9f39-77c6-4cc7-bd62-e1daa76fb080', 'HTTPStatusCode': 202, 'HTTPHeaders': {'date': 'Wed, 18 Feb 2026 04:10:07 GMT', 'content-type': 'application/json', 'content-length': '516', 'connection': 'keep-alive', 'x-amzn-requestid': '991a9f39-77c6-4cc7-bd62-e1daa76fb080', 'x-amz-apigw-id': 'Y9ZBAG06oAMERzg=', 'x-amzn-trace-id': 'Root=1-69953b9f-36535afc03a69e993017c01c'}, 'RetryAttempts': 0}, 'agent': {'agentId': '8LTYFW0JQN', 'agentName': 'DummyAgent', 'agentArn': 'arn:aws:bedrock:us-east-1:546203070602:agent/8LTYFW0JQN', 'instruction': 'You are a dummy agent.Your job is to do nothing', 'agentStatus': 'CREATING', 'foundationModel': 'amazon.nova-micro-v1:0', 'description': 'A dummy agent to test whether agent creation works or not', 'orchestrationType': 'DEFAULT', 'idleSessionTTLInSeconds': 600, 'createdAt': datetime.datetime(2026, 2, 18, 4, 10, 7, 766435, tzinfo=tzutc()), 'updatedAt': datetime.datetime(2026, 2, 18, 4, 10, 7, 766435, tzinfo=tzutc()), 'age